# SQL analytics results

Run each file from `sql/analytics/` in the **Supabase SQL Editor**, then paste the result table into the matching section below. These pasted outputs will feed the final project document.

How to run in Supabase:

1. Open the project → **SQL Editor** → New query
2. Paste the SQL from the file named in the section
3. Run
4. Copy the result table into the **Result** cell
5. Add a short interpretation in **Notes**

Queries expect `core.customers`, `core.customer_usage`, and `core.churn_labels`. If a query fails, paste the error in **Result** so we can match table names.

## 00 — Load checks

File: `sql/analytics/00_load_checks.sql`

Expected roughly: 10,000 customers, ~120,000 usage rows, 10,000 churn labels, 0 missing links.

**Result**

| check_name              | value  |
| ----------------------- | ------ |
| customers               | 10000  |
| customer_usage          | 120060 |
| churn_labels            | 10000  |
| customers_missing_usage | 0      |
| customers_missing_churn | 0      |

**Notes**

_The database contains 10,000 customers, 120,060 usage records, and 10,000 churn labels. No customers are missing linked usage or churn records, indicating complete customer-level coverage across these tables. These checks confirm record counts and relationship coverage_

## 01 — Churn rate by contract

File: `sql/analytics/01_churn_rate_by_contract.sql`

**Result**

| contract_type | customers | churners | churn_rate_pct |
| ------------- | --------- | -------- | -------------- |
| Yearly        | 2006      | 177      | 8.82           |
| Prepaid       | 4041      | 310      | 7.67           |
| Monthly       | 3953      | 299      | 7.56           |

**Notes**

_Yearly contracts have the highest churn rate (8.82%; 177 of 2,006 customers), followed by Prepaid (7.67%) and Monthly (7.56%). The Yearly rate is 1.26 percentage points higher than Monthly, despite Yearly having fewer churners in absolute terms. Business implication: Investigate dissatisfaction or renewal issues among Yearly customers; this is an association, not evidence that contract type causes churn._

## 02 — Churn by tenure bucket

File: `sql/analytics/02_churn_by_tenure_bucket.sql`

**Result**

| tenure_bucket | customers | churners | churn_rate_pct |
| ------------- | --------- | -------- | -------------- |
| 00-11 months  | 894       | 66       | 7.38           |
| 12-23 months  | 1779      | 145      | 8.15           |
| 24-35 months  | 1811      | 147      | 8.12           |
| 36-47 months  | 1725      | 127      | 7.36           |
| 48+ months    | 3791      | 301      | 7.94           |._

**Notes**

_Churn varies only modestly across tenure groups: 12–23 months (8.15%) and 24–35 months (8.12%) are highest, while 36–47 months (7.36%) and 0–11 months (7.38%) are lowest. The 48+ month group has a 7.94% churn rate and the largest number of churners (301), partly reflecting its larger customer base (3,791). Conclusion: The results do not support the idea that the newest customers churn the most; churn risk is relatively similar across tenure buckets._

## 03 — Average usage by churn status

File: `sql/analytics/03_average_usage_by_churn.sql`

**Result**

| churn | avg_call_minutes | avg_data_gb | avg_sms | avg_complaints | customers | usage_rows |
| ----- | ---------------- | ----------- | ------- | -------------- | --------- | ---------- |
| 0     | 492.22           | 9.90        | 24.50   | 0.50           | 9214      | 110616     |
| 1     | 494.75           | 9.87        | 24.84   | 0.51           | 786       | 9444       |

**Notes**

_Churned and retained customers have nearly identical average usage and complaint levels. Churners average 494.75 call minutes vs 492.22 for non-churners, 9.87 GB vs 9.90 GB of data, 24.84 vs 24.50 SMS, and 0.51 vs 0.50 complaints. Conclusion: These group averages show no strong separation by churn status; usage and complaints alone, at this aggregated level, appear insufficient to explain churn. More detailed analysis may reveal differences hidden by averages._

## 04 — Complaints by contract and region

File: `sql/analytics/04_complaints_by_contract_region.sql`

**Result**

| contract_type | region   | complaints | customers | avg_monthly_complaints | total_complaints |
| ------------- | -------- | ---------- | --------- | ---------------------- | ---------------- |
| Yearly        | Rural    | 4342       | 688       | 0.53                   | 4342             |
| Monthly       | Suburban | 8456       | 1384      | 0.51                   | 8456             |
| Prepaid       | Rural    | 8063       | 1325      | 0.51                   | 8063             |
| Prepaid       | Urban    | 8297       | 1373      | 0.50                   | 8297             |
| Monthly       | Rural    | 7735       | 1298      | 0.50                   | 7735             |
| Monthly       | Urban    | 7707       | 1271      | 0.50                   | 7707             |
| Prepaid       | Suburban | 8033       | 1343      | 0.50                   | 8033             |
| Yearly        | Suburban | 3984       | 675       | 0.49                   | 3984             |
| Yearly        | Urban    | 3776       | 643       | 0.49                   | 3776             |

**Notes**

_The largest total complaint counts are Monthly–Suburban (8,456), Prepaid–Urban (8,297), and Prepaid–Rural (8,063). However, group sizes differ, so total complaints should not be interpreted as per-customer risk. Yearly–Rural has the highest average monthly complaints (0.53), while Yearly–Suburban and Yearly–Urban are lowest (0.49). Conclusion: Normalized complaint averages are tightly clustered (0.49–0.53), suggesting limited variation by contract and region in this summary._

## 05 — Revenue by contract and region

File: `sql/analytics/05_revenue_by_contract_region.sql`

**Result**

| contract_type | region   | customers | monthly_revenue | avg_monthly_charges |
| ------------- | -------- | --------- | --------------- | ------------------- |
| Monthly       | Suburban | 1384      | 70906.09        | 51.23               |
| Prepaid       | Urban    | 1373      | 70008.38        | 50.99               |
| Prepaid       | Suburban | 1343      | 68309.07        | 50.86               |
| Prepaid       | Rural    | 1325      | 68164.08        | 51.44               |
| Monthly       | Rural    | 1298      | 66122.36        | 50.94               |
| Monthly       | Urban    | 1271      | 63967.94        | 50.33               |
| Yearly        | Rural    | 688       | 35757.48        | 51.97               |
| Yearly        | Suburban | 675       | 33880.75        | 50.19               |
| Yearly        | Urban    | 643       | 32923.79        | 51.20               |

**Notes**

_Monthly–Suburban contributes the highest monthly revenue (70,906.09), followed by Prepaid–Urban (70,008.38) and Prepaid–Suburban (68,309.07). Yearly–Urban contributes the least (32,923.79). Average monthly charges are similar across segments (approximately 50.19–51.97), so differences in total revenue largely reflect customer counts rather than substantial differences in average charges. Business implication: Larger Monthly and Prepaid segments are major contributors to the current monthly revenue base. Currency is not specified in the results._

## 06 — High-value / high-risk (sample)

File: `sql/analytics/06_high_value_high_risk.sql`

This query returns one row per customer. For the notebook, paste only the **first 15–20 Critical rows**, not the full 10,000.

**Result**

| customer_id | contract_type | region   | monthly_charges | tenure_months | avg_complaints | avg_data_gb | churn | retention_priority |
| ----------- | ------------- | -------- | --------------- | ------------- | -------------- | ----------- | ----- | ------------------ |
| CUST_9404   | Monthly       | Urban    | 369.55          | 50            | 0.58           | 8.30        | 0     | Critical           |
| CUST_3666   | Prepaid       | Urban    | 367.65          | 28            | 0.67           | 9.53        | 0     | Critical           |
| CUST_5772   | Monthly       | Urban    | 356.00          | 6             | 0.58           | 8.79        | 0     | Critical           |
| CUST_8612   | Prepaid       | Urban    | 338.10          | 72            | 0.67           | 10.61       | 0     | Critical           |
| CUST_2155   | Monthly       | Rural    | 318.50          | 32            | 0.25           | 10.59       | 1     | Critical           |
| CUST_5224   | Yearly        | Rural    | 316.80          | 35            | 0.75           | 8.66        | 1     | Critical           |
| CUST_2666   | Monthly       | Suburban | 302.60          | 29            | 0.92           | 7.55        | 0     | Critical           |
| CUST_2597   | Yearly        | Urban    | 298.80          | 63            | 0.67           | 10.06       | 0     | Critical           |
| CUST_1036   | Monthly       | Suburban | 295.55          | 33            | 0.42           | 9.51        | 1     | Critical           |
| CUST_1133   | Prepaid       | Suburban | 281.85          | 6             | 0.58           | 10.69       | 0     | Critical           |
| CUST_5281   | Prepaid       | Rural    | 254.55          | 47            | 0.67           | 11.24       | 0     | Critical           |
| CUST_2438   | Monthly       | Urban    | 248.00          | 58            | 0.75           | 10.30       | 0     | Critical           |
| CUST_1810   | Monthly       | Suburban | 244.00          | 21            | 0.67           | 10.36       | 0     | Critical           |
| CUST_8906   | Yearly        | Rural    | 242.80          | 66            | 0.58           | 9.35        | 0     | Critical           |
| CUST_1830   | Monthly       | Rural    | 217.40          | 61            | 0.67           | 11.74       | 0     | Critical           |
| CUST_3633   | Prepaid       | Rural    | 152.25          | 33            | 0.58           | 9.82        | 0     | Critical           |
| CUST_4162   | Prepaid       | Urban    | 107.14          | 8             | 0.67           | 9.53        | 0     | Critical           |
| CUST_6219   | Monthly       | Suburban | 102.17          | 71            | 0.75           | 8.95        | 0     | Critical           |
| CUST_2422   | Yearly        | Urban    | 101.04          | 26            | 0.50           | 10.56       | 1     | Critical           |
| CUST_9976   | Monthly       | Rural    | 100.79          | 28            | 0.58           | 10.05       | 0     | Critical           |

**Notes**

This is a **sample of the SQL Critical list**, not the later 1,000-customer model tier. The 20 rows are high `monthly_charges` (about 101–370 versus a typical ~51 in query 05), so some of these IDs may be charge outliers as well as high-value accounts. Contracts and regions are mixed. Most sample customers have `churn = 0`; four have `churn = 1` (CUST_2155, CUST_5224, CUST_1036, CUST_2422). Complaint averages in the sample sit around 0.25–0.92. Use query 07 for counts; use this sample only to illustrate who the SQL proxy flags.

## 07 — Retention priority counts

File: `sql/analytics/07_retention_priority_counts.sql`

Use this for the final document instead of the full customer list. Tiers are a SQL proxy (charges + complaints + actual churn), not a machine-learning score.

**Result**

| retention_priority | customers |
| ------------------ | --------- |
| Critical           | 1057      |
| Maintain           | 1445      |
| Standard           | 3363      |
| Low priority       | 4135      |

**Notes**

_Of 10,000 customers, 1,057 (10.57%) are classified as Critical, 1,445 (14.45%) as Maintain, 3,363 (33.63%) as Standard, and 4,135 (41.35%) as Low priority. Critical is the smallest segment, while Low priority is the largest. Business implication: The Critical tier provides a manageable starting group for retention review_